In [11]:
import os
import sys
cur_dir = os.getcwd()
sys.path.append(os.path.join(cur_dir, '..'))

# 데이터 불러오기

In [12]:
import torch
from torch.utils.data import DataLoader
from teds_tensor_dataset import TEDSTensorDataset
from utils.processing_utils import train_test_split_customed_dataset

root = os.path.join(cur_dir, '..', 'data_tensor_cache')
dataset = TEDSTensorDataset(root)

_, _, test_dataset = train_test_split_customed_dataset(dataset=dataset, seed=42)

col_list, col_dims, ad_col_index, dis_col_index = dataset.col_info

저장되어 있는 전처리된 데이터가 있습니다. 해당 데이터를 불러오는 중..
불러오기 완료
Train Set Size: 975896
Valid Set Size: 209119
Test Set Size: 209123


# 하이퍼파라미터

In [13]:
batch_size = 1
embedding_dim = 32
gin_hidden_channel = 32
gin_layers = 2
gru_hidden_channel = 64

scheduler_patience = 15
device = torch.device("cpu")

epochs=100
lr = 0.001

# 모델 불러오기

In [14]:
from models.gingru_for_explain import GinGruForExplain

model = GinGruForExplain(
    batch_size=batch_size,
    col_list=col_list,
    col_dims=col_dims,
    ad_col_index=ad_col_index,
    dis_col_index=dis_col_index,
    embedding_dim=embedding_dim,
    gin_hidden_channel=gin_hidden_channel,
    train_eps=True,
    gin_layers=gin_layers,
    gru_hidden_channel=gru_hidden_channel
)

model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, patience=scheduler_patience)
checkpoint_path = os.path.join(cur_dir, '..', 'checkpoints', 'gingru', 'real', 'best_gingru_epoch_6_loss_0.3761.pth')

In [15]:

def load_checkpoint(model, optimizer, scheduler, filename, device):
    """
    저장된 체크포인트(.pth)를 불러와서 
    model, optimizer, scheduler 상태를 복구합니다.

    Parameters:
        model (nn.Module): 모델 객체
        optimizer (torch.optim.Optimizer): 옵티마이저 객체
        scheduler: 스케줄러 객체
        filename (str): 저장된 체크포인트 경로
        device: CPU로 로드하고 싶으면 torch.device('cpu')

    Returns:
        start_epoch (int): 다음 훈련을 시작할 epoch 번호
        best_loss (float): 저장된 최소 validation loss
    """
    checkpoint = torch.load(filename, map_location=device)

    # --- Load states ---
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint['best_loss']

    return start_epoch, best_loss

start_epoch, best_loss = load_checkpoint(model=model,
                                         optimizer=optimizer,
                                         scheduler=scheduler,
                                         filename=checkpoint_path,
                                         device=device)

model

GinGruForExplain(
  (entity_embedding_layer): EntityEmbeddingBatch3(
    (embedding_layer): Embedding(1143, 32)
  )
  (gin_layers): ModuleList(
    (0-1): 2 x GINConv(nn=Sequential(
      (0): Linear(in_features=32, out_features=32, bias=True)
      (1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=32, out_features=32, bias=True)
    ))
  )
  (gru_layer): GRU(64, 64)
  (classifier_b): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [16]:
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import DummyExplainer, GNNExplainer
from torch_geometric.explain.config import ModelConfig

model_config = ModelConfig(
    mode='binary_classification',
    task_level="graph",
    return_type="raw"
)
#########여기서부터 다시 하면 됨
algorithm = GNNExplainer(
    epochs=epochs,
    lr=lr
)

explainer = Explainer(
    model=model,
    algorithm=algorithm,
    explanation_type="phenomenon",
    model_config=model_config,
    node_mask_type=None,
    edge_mask_type="object",
    threshold_config=None
)


## 예시

In [22]:
from utils.processing_utils import mi_edge_index_improved
mi_dict_path = os.path.join(root, 'data', 'mi_dict_static.pickle')
edge_index = mi_edge_index_improved(mi_dict_path)
num_nodes = len(ad_col_index)
dis_edge_index = edge_index + num_nodes
edge_index = torch.cat((edge_index, dis_edge_index), dim=1)

x, y, los = test_dataset[0]
x_embedded = model.entity_embedding_layer(x)

# 필요한 인덱스 정보 (클래스 내부에서 가져와야 함)
ad_idx_t = model.ad_idx_t
dis_idx_t = model.dis_idx_t

x_ad_nodes = torch.index_select(x_embedded.detach(), dim=0, index=ad_idx_t) # [60, F]
x_dis_nodes = torch.index_select(x_embedded.detach(), dim=0, index=dis_idx_t) # [60, F]
x_embedded = torch.cat([x_ad_nodes, x_dis_nodes], dim=0).detach() # [120, F]

algorithm = GNNExplainer(
    epochs=epochs,
    lr=lr
)

explainer = Explainer(
    model=model,
    algorithm=algorithm,
    explanation_type="phenomenon",
    model_config=model_config,
    node_mask_type=None,
    edge_mask_type="object",
    threshold_config=None
)


# 4. Explainer 실행
explanation = explainer(x=x_embedded, edge_index=edge_index, target=y, index=None, los=los, device=device)

del explainer
del algorithm
import gc
gc.collect()

# explanation.edge_mask 확인

0

In [23]:
explanation.edge_mask

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 

In [24]:
from utils.processing_utils import mi_edge_index_improved
from torch_geometric.explain import ExplainerConfig

explainer_config = ExplainerConfig(
    explanation_type="phenomenon",
    node_mask_type=None,
    edge_mask_type="object"
)

# 1. 데이터 및 엣지 준비 (기존 코드)
mi_dict_path = os.path.join(root, 'data', 'mi_dict_static.pickle')
edge_index = mi_edge_index_improved(mi_dict_path)
num_nodes = len(ad_col_index)
dis_edge_index = edge_index + num_nodes
edge_index = torch.cat((edge_index, dis_edge_index), dim=1)

x, y, los = test_dataset[0]

x_embedded = model.entity_embedding_layer(x)
ad_idx_t = model.ad_idx_t
dis_idx_t = model.dis_idx_t
x_ad_nodes = torch.index_select(x_embedded.detach(), dim=0, index=ad_idx_t)
x_dis_nodes = torch.index_select(x_embedded.detach(), dim=0, index=dis_idx_t)
x_embedded = torch.cat([x_ad_nodes, x_dis_nodes], dim=0).detach() # [120, F]

# ----------------------------------------------------
# [해결책] Explainer 알고리즘을 매번 새로 생성하여 초기화를 강제
# ----------------------------------------------------

# 2. GNNExplainer 알고리즘 재정의 (마스크 변수 초기화)
algorithm = GNNExplainer(
    epochs=epochs,
    lr=lr
)

# 3. Explainer Wrapper 객체 전체를 새로 생성
explainer = Explainer(
    model=model,
    algorithm=algorithm, # 새로 만든 알고리즘 객체 사용
    explanation_type="phenomenon",
    model_config=model_config,
    node_mask_type=None,
    edge_mask_type="object",
    threshold_config=None
)

# 4. Explainer 실행
explanation = explainer(x=x_embedded, edge_index=edge_index, target=y, index=None, los=los, device=device)

# explanation.edge_mask 확인

In [25]:
explanation.edge_mask

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 